# JEPA Stage B writeup: b01-b02 results and revised next steps

This notebook follows `02-writeup-7-27-health.ipynb` and
`03-writeup-7-27-downstream.ipynb`. It reconstructs the Stage A premise from the
portable result ledger, validates the two available 100-epoch Stage B runs, and keeps
the later legacy GFC analysis explicitly separate from the revised GFC-v2 proposal.

The central question is whether Stage B produced one model that combined the two
properties separated in Stage A: useful downstream features and a broad,
context-sensitive pooled representation.


In [ ]:
from pathlib import Path
import json
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'results').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the cody-jepa repository root')


REPO_ROOT = find_repo_root()
IMAGE_DIR = Path(os.environ.get(
    'CODY_JEPA_WRITEUP_IMAGE_DIR', REPO_ROOT / 'docs' / 'images'
)).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get(
    'CODY_JEPA_REPRO_OUTPUT_DIR',
    REPO_ROOT / 'results' / 'generated' / 'writeup-stage-b',
)).expanduser().resolve()
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_PATH = REPO_ROOT / 'results' / 'checkpoint_histories.csv'
METADATA_PATH = REPO_ROOT / 'results' / 'checkpoint_histories.json'
SUMMARY_PATH = REPO_ROOT / 'results' / 'phase1_summary.csv'
required = [HISTORY_PATH, METADATA_PATH, SUMMARY_PATH]
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing compact result inputs: {missing}')

STAGE_A_RUNS = [
    'a00-baseline', 'a01-lr3e-5', 'a02-lr3e-4', 'a03-ema0.995',
    'a04-mask-light', 'a05-mask-heavy', 'a06-pred-depth3', 'a07-clip-var',
]
STAGE_B_RUNS = ['b01-mask-light', 'b02-mask-light-clip-var']
FOCUS_RUNS = [
    'a00-baseline', 'a04-mask-light', 'a07-clip-var',
    'b01-mask-light', 'b02-mask-light-clip-var',
]
RUN_LABELS = {
    'a00-baseline': 'a00 baseline',
    'a04-mask-light': 'a04 light mask',
    'a07-clip-var': 'a07 clip variance',
    'b01-mask-light': 'b01 light mask',
    'b02-mask-light-clip-var': 'b02 light mask + clip variance',
}

print(f'Repository: {REPO_ROOT}')
print(f'Writeup images: {IMAGE_DIR}')
print(f'Exact table values: {OUTPUT_DIR}')


## Experiment contract and lineage

The attached 7/27 status proposed three 100-epoch checks: clip variance alone, light
masking alone, and their combination. The portable results contain **b01** and **b02**,
but no b00 result. Therefore b01 tests whether the a04 light-mask behavior persists at
the longer budget, while b02 tests the combination. The planned independent long-run
confirmation of a07 is unavailable and is not inferred from b02.


In [ ]:
history_metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
run_metadata = {row['run_id']: row for row in history_metadata['runs']}

for run_id in STAGE_A_RUNS:
    metadata = run_metadata[run_id]
    if metadata['completed_epochs'] != 40 or metadata['global_step'] != 1560:
        raise AssertionError(f'{run_id}: expected the matched Stage A budget')

for run_id in STAGE_B_RUNS:
    metadata = run_metadata[run_id]
    if metadata['completed_epochs'] != 100 or metadata['global_step'] != 3900:
        raise AssertionError(f'{run_id}: expected the matched Stage B budget')
    if metadata['config']['mask_preset'] != 'light':
        raise AssertionError(f'{run_id}: expected light masking')

if run_metadata['b01-mask-light']['config']['clip_var_coef'] != 0.0:
    raise AssertionError('b01 must isolate light masking without clip variance')
if run_metadata['b02-mask-light-clip-var']['config']['clip_var_coef'] != 1.0:
    raise AssertionError('b02 must combine light masking with clip variance')

contract = pd.DataFrame([
    {
        'Run': 'b01', 'Configuration': 'light mask', 'Epochs': 100,
        'Optimizer steps': 3900, 'Stage A antecedent': 'a04 light mask',
        'Question': 'Does the downstream-preserving light-mask result persist?',
    },
    {
        'Run': 'b02', 'Configuration': 'light mask + clip variance', 'Epochs': 100,
        'Optimizer steps': 3900, 'Stage A antecedent': 'a04 + a07',
        'Question': 'Can breadth/context sensitivity and downstream utility coexist?',
    },
])
print(contract.to_string(index=False))
print()
print('No b00 compact result is present; a07 lacks its planned independent 100-epoch check.')


## Load and validate selected-checkpoint results

Stage A checkpoints were selected from 40-epoch runs. Stage B checkpoints were selected
from 100-epoch runs: b01 by best subject-balanced loss and b02 by the stored healthy
checkpoint rule. The selected values below are cross-checked against the full portable
history rather than trusted as prose.


In [ ]:
summary = pd.read_csv(SUMMARY_PATH, float_precision='round_trip').set_index('run_id')
history = pd.read_csv(HISTORY_PATH, float_precision='round_trip')

metric_columns = [
    'selected_epoch', 'validation_loss', 'effective_rank', 'effective_rank_ratio',
    'wrong_context_gap', 'relative_gap', 'closed_set_identity_accuracy',
    'held_out_retrieval_accuracy', 'gait_balanced_accuracy',
]
selected = summary.loc[FOCUS_RUNS, metric_columns].copy()
selected.insert(0, 'label', [RUN_LABELS[run_id] for run_id in selected.index])

history_metric_map = {
    'validation_loss': 'val_loss',
    'effective_rank': 'val_effective_rank',
    'wrong_context_gap': 'val_subject_balanced_context_shuffle_loss_gap',
}
for run_id, row in selected.iterrows():
    observed = history.loc[
        history['run_id'].eq(run_id)
        & history['epoch'].eq(int(row['selected_epoch']))
        & history['val_loss'].notna()
    ]
    if len(observed) != 1:
        raise ValueError(f'{run_id}: selected epoch missing or duplicated in history')
    observed = observed.iloc[0]
    for summary_key, history_key in history_metric_map.items():
        if not np.isclose(row[summary_key], observed[history_key], rtol=1e-9, atol=1e-12):
            raise AssertionError(f'{run_id}: {summary_key} disagrees with history')

selected_path = OUTPUT_DIR / 'stage-b-selected-checkpoint-values.csv'
selected.to_csv(selected_path, float_format='%.17g')
display_columns = selected.copy()
for column in [
    'effective_rank_ratio', 'relative_gap', 'closed_set_identity_accuracy',
    'held_out_retrieval_accuracy', 'gait_balanced_accuracy',
]:
    display_columns[column] = display_columns[column].map(lambda value: f'{100 * value:.2f}%')
display(display_columns)
print(f'Exact selected-checkpoint values: {selected_path}')


## Stage B training trajectories

Both runs are shown at every stored two-epoch evaluation. The marked checkpoint is the
one used by the downstream and legacy GFC summaries. The endpoints are also important:
b01 and b02 had already plateaued near their selected values, so the qualitative split
is not an artifact of choosing one isolated epoch.


In [ ]:
trajectory_columns = {
    'val_loss': ('JEPA validation loss', False),
    'val_cosine': ('Cosine similarity', False),
    'val_effective_rank': ('Online full-view effective rank', False),
    'val_subject_balanced_context_shuffle_loss_gap': (
        'Subject-balanced wrong-context loss gap', True,
    ),
}
stage_b_history = history.loc[
    history['run_id'].isin(STAGE_B_RUNS) & history['val_loss'].notna()
].copy()
if stage_b_history.groupby('run_id').size().ne(50).any():
    raise ValueError('Each Stage B run must contribute 50 two-epoch validation rows')
if stage_b_history.groupby('run_id')['epoch'].max().ne(100).any():
    raise ValueError('Each Stage B trajectory must reach epoch 100')

trajectory_path = OUTPUT_DIR / 'stage-b-health-trajectories.csv'
stage_b_history[['run_id', 'epoch', 'step', *trajectory_columns]].to_csv(
    trajectory_path, index=False, float_format='%.17g'
)

colors = {'b01-mask-light': '#4c78a8', 'b02-mask-light-clip-var': '#f28e2b'}
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2), constrained_layout=True)
for ax, (metric, (title, zero_line)) in zip(axes.flat, trajectory_columns.items()):
    for run_id in STAGE_B_RUNS:
        rows = stage_b_history.loc[stage_b_history['run_id'].eq(run_id)]
        ax.plot(
            rows['step'], rows[metric], color=colors[run_id], linewidth=2,
            label=RUN_LABELS[run_id],
        )
        selected_epoch = int(summary.loc[run_id, 'selected_epoch'])
        point = rows.loc[rows['epoch'].eq(selected_epoch)].iloc[0]
        ax.scatter(point['step'], point[metric], color=colors[run_id], s=45, zorder=3)
    if zero_line:
        ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('optimizer step')
    ax.grid(True, alpha=0.25)
axes[0, 0].legend(frameon=False)
fig.suptitle('Stage B training health: 100 epochs, seed 0', fontsize=15)
trajectory_image = IMAGE_DIR / 'stage-b-health-trajectories.png'
fig.savefig(trajectory_image, dpi=180, bbox_inches='tight')
plt.show()
print(trajectory_image)
print(f'Exact plotted values: {trajectory_path}')


### Trajectory reading

- **b01 optimizes but remains narrow.** Loss falls steadily to about 0.382, while
  effective rank plateaus near 11 of 384 and the wrong-context gap remains around
  0.00035.
- **b02 changes the representation regime.** Effective rank rises to about 76 and the
  wrong-context gap to about 0.113, while loss settles near 0.555. The clip-variance
  penalty therefore does what it was designed to do at the pooled-feature level.
- **The tradeoff persists.** Longer training does not make b01 acquire b02-like breadth,
  and it does not make b02 converge to b01-like loss.

These statements concern the historical substitution diagnostic. The revised proposal
uses a normalized, geometry-matched intervention on held-out GaitLU sequences; the raw
Health&Gait wrong-context gap is not that planned outcome.


## Selected-checkpoint comparison to the Stage A premise


In [ ]:
plot_specs = [
    ('validation_loss', 'Validation loss', 1.0),
    ('effective_rank_ratio', 'Effective-rank ratio', 100.0),
    ('relative_gap', 'Relative wrong-context gap', 100.0),
    ('closed_set_identity_accuracy', 'Closed-set identity accuracy', 100.0),
    ('held_out_retrieval_accuracy', 'Held-out identity retrieval', 100.0),
    ('gait_balanced_accuracy', 'Speed balanced accuracy', 100.0),
]
plot_runs = ['a00-baseline', 'a04-mask-light', 'a07-clip-var', *STAGE_B_RUNS]
plot_colors = ['#8c8c8c', '#59a14f', '#e15759', '#4c78a8', '#f28e2b']
fig, axes = plt.subplots(2, 3, figsize=(14, 8.2), constrained_layout=True)
for ax, (metric, title, scale) in zip(axes.flat, plot_specs):
    values = selected.loc[plot_runs, metric].to_numpy() * scale
    ax.barh(range(len(plot_runs)), values, color=plot_colors)
    ax.set_yticks(range(len(plot_runs)), [RUN_LABELS[run_id] for run_id in plot_runs])
    ax.invert_yaxis()
    ax.set_title(title)
    ax.margins(x=0.20)
    ax.grid(True, axis='x', alpha=0.25)
    suffix = '%' if scale == 100.0 else ''
    for index, value in enumerate(values):
        ax.text(value, index, f' {value:.2f}{suffix}', va='center', fontsize=8)
fig.suptitle('Stage A antecedents and Stage B selected checkpoints', fontsize=15)
comparison_image = IMAGE_DIR / 'stage-b-selected-checkpoints.png'
fig.savefig(comparison_image, dpi=180, bbox_inches='tight')
plt.show()
print(comparison_image)


### What Stage B established

Relative to a04, b01 lowers selected validation loss from 0.4912 to 0.3823, but its
effective-rank ratio is still only 2.83% and its relative wrong-context gap is 0.092%.
Its closed-set identity result is essentially unchanged, held-out retrieval improves by
only 0.18 percentage points, and speed balanced accuracy falls by 1.23 points.

Relative to a07, b02 nearly triples pooled effective rank (25.2 to 75.2), more than
doubles the raw wrong-context gap (0.0523 to 0.1136), and raises held-out identity
retrieval from 4.04% to 4.84%. It also reduces closed-set identity accuracy to 5.79%,
reduces speed balanced accuracy to 88.41%, and has higher prediction loss.

Thus b02 validates a **mechanistic repair** for pooled breadth and this context
intervention, not a generally superior representation. Stage B did not produce a
single checkpoint that dominates across optimization, breadth, context sensitivity,
factor decoding, and identity.


## Legacy GFC development result

The completion summaries checked into `results/gfc-*` were generated later than the
7/27 status. They use `legacy_donor_excluded_v1`: 24 queries per complete participant,
both donors removed from a six-cell gallery, 308 complete historical training
participants for fitting, and 76 complete development participants for evaluation.
They are useful for diagnosing the Stage B tradeoff, but they are not GFC-v2 outcomes.


In [ ]:
normalization_order = ['raw_retain_all', 'raw_effective_rank', 'pca_effective_rank']
gfc_rows = []
for result_dir in sorted((REPO_ROOT / 'results').glob('gfc-*')):
    path = result_dir / 'summary.json'
    if not path.is_file():
        continue
    payload = json.loads(path.read_text(encoding='utf-8'))
    if payload['model_label'] not in ['a00-baseline', *STAGE_B_RUNS]:
        continue
    if payload['protocol'] != 'legacy_donor_excluded_v1':
        raise AssertionError(f'{path}: unexpected protocol')
    if payload['evaluation']['participant_count'] != 76:
        raise AssertionError(f'{path}: unexpected evaluation cohort')
    if payload['method_settings']['gallery']['size'] != 6:
        raise AssertionError(f'{path}: legacy gallery must have six cells')
    if payload['method_settings']['query']['queries_per_participant'] != 24:
        raise AssertionError(f'{path}: legacy protocol must have 24 queries')
    interval = payload['learned_minus_shortcut']['confidence_interval']
    gfc_rows.append({
        'run_id': payload['model_label'],
        'normalization': payload['normalization'],
        'learned_top1': payload['learned']['top1'],
        'shortcut_top1': payload['shortcut']['top1'],
        'learned_minus_shortcut': payload['learned_minus_shortcut']['point_estimate'],
        'ci_lower': interval['lower'], 'ci_upper': interval['upper'],
        'learned_mrr': payload['learned']['mrr'],
        'learned_donor_attraction': payload['learned']['donor_attraction'],
    })

gfc = pd.DataFrame(gfc_rows)
if len(gfc) != 9 or gfc.groupby('run_id').size().ne(3).any():
    raise ValueError('Expected three legacy normalizations for a00, b01, and b02')
gfc['normalization'] = pd.Categorical(
    gfc['normalization'], normalization_order, ordered=True
)
gfc = gfc.sort_values(['run_id', 'normalization']).reset_index(drop=True)
gfc_path = OUTPUT_DIR / 'stage-b-legacy-gfc-values.csv'
gfc.to_csv(gfc_path, index=False, float_format='%.17g')
display(gfc)
print(f'Exact legacy GFC values: {gfc_path}')


In [ ]:
primary = gfc.loc[gfc['normalization'].eq('raw_retain_all')].set_index('run_id').loc[
    ['a00-baseline', *STAGE_B_RUNS]
]
fig, (ax_top1, ax_gap) = plt.subplots(1, 2, figsize=(13, 5.4), constrained_layout=True)

x = np.arange(len(primary))
width = 0.36
ax_top1.bar(x - width / 2, 100 * primary['learned_top1'], width, label='learned', color='#4c78a8')
ax_top1.bar(x + width / 2, 100 * primary['shortcut_top1'], width, label='shortcut', color='#bab0ac')
ax_top1.set_xticks(x, [RUN_LABELS[run_id] for run_id in primary.index], rotation=15, ha='right')
ax_top1.set_ylabel('top-1 (%)')
ax_top1.set_title('Declared historical normalization: raw retain-all')
ax_top1.legend(frameon=False)
ax_top1.grid(True, axis='y', alpha=0.25)

markers = ['o', 's', '^']
offsets = [-0.20, 0.0, 0.20]
for normalization, marker, offset in zip(normalization_order, markers, offsets):
    rows = gfc.loc[gfc['normalization'].eq(normalization)].set_index('run_id').loc[primary.index]
    center = 100 * rows['learned_minus_shortcut']
    lower = center - 100 * rows['ci_lower']
    upper = 100 * rows['ci_upper'] - center
    ax_gap.errorbar(
        x + offset, center, yerr=np.vstack([lower, upper]), fmt=marker,
        capsize=3, linewidth=1.3, label=normalization,
    )
ax_gap.axhline(0, color='black', linewidth=1)
ax_gap.set_xticks(x, [RUN_LABELS[run_id] for run_id in primary.index], rotation=15, ha='right')
ax_gap.set_ylabel('learned - shortcut top-1 (percentage points)')
ax_gap.set_title('Normalization sensitivity with 95% participant bootstrap')
ax_gap.legend(frameon=False, fontsize=8)
ax_gap.grid(True, axis='y', alpha=0.25)

fig.suptitle('Legacy donor-excluded development result - not GFC-v2', fontsize=15)
legacy_image = IMAGE_DIR / 'stage-b-legacy-gfc.png'
fig.savefig(legacy_image, dpi=180, bbox_inches='tight')
plt.show()
print(legacy_image)


### Legacy GFC reading

Under the declared historical `raw_retain_all` normalization, a00 learned top-1 was
69.79% versus 65.46% for the shortcut (+4.33 percentage points, 95% participant
bootstrap [+0.27, +8.22]). b01 scored 63.76% (-1.70 points versus shortcut, interval
crossing zero), and b02 scored 57.51% (-7.95 points, interval entirely below zero).

The qualitative b02 failure persists under all three normalizations; b01 never
separates positively from the shortcut. At the same time, normalization materially
changes a00's estimated advantage. This is evidence that diagnostic breadth and
held-out identity retrieval did not predict legacy completion performance across these
three checkpoints. It is not a model-level population estimate, a data-scaling result,
or evidence about GFC-v2.


## Revised next steps


In [ ]:
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['svg.hashsalt'] = 'cody-jepa-stage-b-revised-next-steps-v1'
fig, ax = plt.subplots(figsize=(13, 7.2), constrained_layout=True)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('From historical Stage B to the prospectively locked scaling study', fontsize=16, pad=18)

steps = [
    (0.04, 0.70, 0.26, 0.18, 'Historical evidence\nStage A/B + legacy GFC\nMotivation only', '#e8f1fb', '#4c78a8'),
    (0.37, 0.70, 0.26, 0.18, 'Freeze the instrument\nGFC-v2 + controls\nOracle and session tests', '#eaf5e7', '#59a14f'),
    (0.70, 0.70, 0.26, 0.18, 'Prepare GaitLU\nDedupe + source groups\n4 nested pools x 5 seeds', '#fff0df', '#f28e2b'),
    (0.70, 0.29, 0.26, 0.18, 'Train 20 runs\nFixed exposure and config\nFinal checkpoint primary', '#fff0df', '#f28e2b'),
    (0.37, 0.29, 0.26, 0.18, 'Freeze analysis\nSettings + templates\nKeep outcomes locked', '#eaf5e7', '#59a14f'),
    (0.04, 0.29, 0.26, 0.18, 'Open outcomes once\nRun-level scale contrast\nPositive / equivalent / unresolved', '#fdeaea', '#e15759'),
]

def add_box(spec):
    x0, y0, width0, height0, text0, face, edge = spec
    patch = FancyBboxPatch(
        (x0, y0), width0, height0,
        boxstyle='round,pad=0.018,rounding_size=0.018',
        linewidth=1.5, edgecolor=edge, facecolor=face,
    )
    ax.add_patch(patch)
    ax.text(x0 + width0 / 2, y0 + height0 / 2, text0, ha='center', va='center',
            fontsize=10.5, linespacing=1.45)

for spec in steps:
    add_box(spec)

def edge_center(spec, side):
    x0, y0, width0, height0, *_ = spec
    points = {
        'left': (x0, y0 + height0 / 2), 'right': (x0 + width0, y0 + height0 / 2),
        'top': (x0 + width0 / 2, y0 + height0), 'bottom': (x0 + width0 / 2, y0),
    }
    return points[side]

links = [
    (0, 'right', 1, 'left'), (1, 'right', 2, 'left'), (2, 'bottom', 3, 'top'),
    (3, 'left', 4, 'right'), (4, 'left', 5, 'right'),
]
for start, start_side, end, end_side in links:
    ax.add_patch(FancyArrowPatch(
        edge_center(steps[start], start_side), edge_center(steps[end], end_side),
        arrowstyle='-|>', mutation_scale=14, linewidth=1.5, color='#6b7280',
        shrinkA=5, shrinkB=5,
    ))

ax.text(
    0.5, 0.10,
    'Do not select a revised-study configuration from the locked outcome cohort or relabel legacy scores as GFC-v2.',
    ha='center', color='#7f1d1d', fontsize=10.5,
)
roadmap_svg = IMAGE_DIR / 'stage-b-revised-next-steps.svg'
roadmap_png = IMAGE_DIR / 'stage-b-revised-next-steps.png'
fig.savefig(roadmap_svg, bbox_inches='tight', metadata={'Title': 'Stage B revised next steps', 'Date': None})
fig.savefig(roadmap_png, dpi=180, bbox_inches='tight')
plt.show()
print(roadmap_svg)
print(roadmap_png)


The revised proposal changes the scientific question rather than adding another
Health&Gait hyperparameter sweep:

1. **Finish and freeze GFC-v2.** Keep all eight gallery cells, enforce
   `source_video_id` separation, use fractional ties, match the learned and cue paths,
   and include hard/soft factor-completion controls and oracle-spectrum tests.
2. **Prepare GaitLU-only pretraining.** Validate decoding, deduplicate, preserve source
   groups, reserve the common 10,000-sequence context holdout, and materialize five
   seeded nested ladders at about 2.5K, 25K, 250K, and the eligible maximum.
3. **Lock exposure and configuration before outcomes.** Apply one architecture, mask
   policy, objective, augmentation policy, and budget to all 20 runs. Horizontal
   flipping must be disabled because direction is evaluated. The final checkpoint is
   primary.
4. **Freeze the analysis.** Record the protocol version, gallery policy, query count,
   templates, thresholds, and exclusions before opening outcome aggregates.
5. **Open the locked Health&Gait outcomes once.** Fit adapters only on the 80-person
   development cohort and report GFC-v2, probes, normalized GaitLU context reliance,
   pooled/token rank, and the fixed cross-condition identity protocol on the locked
   cohort.
6. **Make the run-level decision.** The primary estimand is full-minus-small learned
   GFC-v2 top-1 across five ladders. Use the prespecified 6.25-point resolution and call
   the result meaningful positive, positive but small, equivalent, or inconclusive.

Stage B's durable contribution is therefore diagnostic: it shows why no single health
metric can choose the reference model and why the revised study must measure
recombination, context reliance, breadth, probes, and identity together.


## Claims boundary

The checked-in evidence supports three conclusions: the light-mask and clip-variance
interventions lead to different representation regimes; clip variance repairs pooled
breadth and increases the historical context-substitution response; and neither b01 nor
b02 improves the legacy completion contrast over a00. It does **not** establish that
more unique GaitLU data helps, that GFC-v2 succeeds, that the representation is causally
disentangled, or that any model is suitable for identity-sensitive deployment.
